In [ ]:
# ============================================================
# 📘 LSTM Model for Time Series Forecasting (Notebook Version)
# ============================================================

# --- Imports ---
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

PyTorch version: 2.9.0+cpu
Using device: cpu


In [14]:
# --- Print environment info ---
print("PyTorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

PyTorch version: 2.9.0+cpu
Using device: cpu


In [ ]:
# ============================================================
# 🔹 Data Handling
# ============================================================


class TrainTestSet:
    def __init__(
        self, name, train_set, test_set, input_window_size, forecast_horizon_size
    ):
        self.name = name
        self.input_window_size = input_window_size
        self.forecast_horizon_size = forecast_horizon_size
        self.train_set = train_set
        self.test_set = test_set

    def get_number_of_train_windows(self):
        return len(self.train_set)


class TimeSeriesDataset(Dataset):
    """Convert sliding window DataFrames to tensors."""

    def __init__(self, windows, input_window, horizon):
        self.X, self.y = [], []

        for window in windows:
            data = window.values
            self.X.append(data[:input_window, :-1])  # features
            self.y.append(data[input_window:, -1])  # target "close"

        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [10]:
# ============================================================
# 🔹 LSTM Model Definition
# ============================================================


class LSTMForecastModel(nn.Module):
    def __init__(self, num_features, hidden_size, num_layers, forecast_horizon):
        super(LSTMForecastModel, self).__init__()
        self.lstm = nn.LSTM(
            input_size=num_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2,
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.ReLU(), nn.Linear(64, forecast_horizon)
        )

    def forward(self, x):
        output, _ = self.lstm(x)  # (batch, seq_len, hidden_size)
        last_output = output[:, -1, :]  # take last time step
        return self.fc(last_output)

In [11]:
# ============================================================
# 🔹 Training Function (with tqdm)
# ============================================================


def train_lstm_model(train_test_set, epochs=30, batch_size=16, lr=1e-3):
    """Train and evaluate the LSTM model."""
    train_dataset = TimeSeriesDataset(
        train_test_set.train_set,
        train_test_set.input_window_size,
        train_test_set.forecast_horizon_size,
    )
    test_dataset = TimeSeriesDataset(
        [train_test_set.test_set],
        train_test_set.input_window_size,
        train_test_set.forecast_horizon_size,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    num_features = train_dataset.X.shape[-1]
    model = LSTMForecastModel(
        num_features,
        hidden_size=128,
        num_layers=2,
        forecast_horizon=train_test_set.forecast_horizon_size,
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    print(f"Training on {device} for {epochs} epochs...\n")

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

        for X_batch, y_batch in progress_bar:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())

        avg_loss = running_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{epochs}] - Avg Train Loss: {avg_loss:.6f}")

    # --- Evaluate ---
    model.eval()
    with torch.no_grad():
        test_losses = []
        for X_batch, y_batch in tqdm(test_loader, desc="Evaluating", leave=False):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            test_losses.append(criterion(y_pred, y_batch).item())

    print(f"\n✅ Test MSE: {np.mean(test_losses):.6f}")
    return model

In [12]:
# ============================================================
# 🔹 Example Usage (Dummy Data)
# ============================================================

# Example configuration
num_features = 220 + 1  # 220 features + "close"
input_window = 360
forecast_horizon = 30

# Create dummy train/test sets
train_windows = [
    pd.DataFrame(
        np.random.rand(input_window + forecast_horizon, num_features),
        columns=[f"f{i}" for i in range(num_features - 1)] + ["close"],
    )
    for _ in range(80)
]

test_window = pd.DataFrame(
    np.random.rand(input_window + forecast_horizon, num_features),
    columns=[f"f{i}" for i in range(num_features - 1)] + ["close"],
)

dataset = TrainTestSet(
    name="example",
    train_set=train_windows,
    test_set=test_window,
    input_window_size=input_window,
    forecast_horizon_size=forecast_horizon,
)

In [1]:
trained_model = train_lstm_model(dataset, epochs=10, batch_size=8)

NameError: name 'train_lstm_model' is not defined